# P1 · Evaluación 01 — Limpieza y EDA de reseñas de Age of Empires II: DE

**Curso:** Métodos Computacionales para la Estadística · Doctorado en Ciencia de Datos, Universidad Bernardo O'Higgins  
**Estudiante:** Sebastián Ulloa  
**Entrega:** P1, Unidad 1 · octubre de 2026

El objetivo es auditar el archivo `aoe2_resenas_p1.csv`, documentar cada problema, construir un dataset limpio y responder las dos preguntas de Genovencio. El notebook está ordenado según la guía: cargar y mirar, funciones, EDA, imputación, codificación, comparación antes/después, correlación, respuesta y guardado.

> **Criterio general:** no se borran filas por un valor imposible ni se rellena un nulo sin justificarlo. Cuando un valor es imposible se anula ese valor y se conserva el resto de la fila; cuando el dato es el objeto de la pregunta (por ejemplo, `recomienda`) no se inventa.

In [ ]:
import sys
import platform
import os
import warnings

warnings.filterwarnings('ignore', message='invalid value encountered in subtract', category=RuntimeWarning)
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import openpyxl
from IPython.display import display

print('Python:', sys.version.split()[0])
print('Sistema:', platform.platform())
print('pandas:', pd.__version__)
print('numpy:', np.__version__)
print('matplotlib:', matplotlib.__version__)
print('seaborn:', sns.__version__)
print('openpyxl:', openpyxl.__version__)

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 180)
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.dpi'] = 110

FECHA_LANZAMIENTO = pd.Timestamp('2019-11-14')
FECHA_DESCARGA = pd.Timestamp('2026-10-11')
MAX_HORAS = (FECHA_DESCARGA - FECHA_LANZAMIENTO).days * 24
MAX_HORAS_2_SEMANAS = 14 * 24
print(f'Horas máximas físicas desde el lanzamiento: {MAX_HORAS} h')

In [ ]:
RUTA = Path('aoe2_resenas_p1.csv')
if not RUTA.exists():
    raise FileNotFoundError(f'No se encontró {RUTA.resolve()}. Sube el CSV a la misma carpeta del notebook.')

df_original = pd.read_csv(RUTA)
df = df_original.copy(deep=True)

print(f'Dimensiones originales: {df_original.shape[0]} filas x {df_original.shape[1]} columnas')
display(df.head(10))
print('\n--- info() ---')
df.info()
print('\n--- describe(include="all") ---')
display(df.describe(include='all').T)

**Lectura inicial.** El archivo tiene **3.938 filas y 9 columnas**, tal como anuncia el enunciado. `head()` muestra filas aparentemente normales, pero `info()` ya revela dos señales: `votos_utiles` está guardada como `object` (texto) y `resenas_autor` como `float64` aunque debería ser un entero. `describe()` confirma valores imposibles: horas totales negativas (mínimo **-635**) y un máximo de **151.781 h**, además de `porcentaje_reciente` con `inf`.

Los datos vienen sucios en varios frentes: 12 formas distintas de escribir `idioma`, 10 de `recomienda` y 17 de `perfil`; hay 60 duplicados exactos y 80 filas extra que repiten `id_resena`; hay nulos explícitos (`NaN`) y nulos disfrazados como `"sin dato"` en `perfil` y `votos_utiles`.

In [ ]:
print('Duplicados exactos:', int(df.duplicated().sum()))
print('Filas extra que repiten id_resena:', int(df.duplicated(subset=['id_resena']).sum()))
print('id_resena únicos:', df['id_resena'].nunique())

print('\nNulos explícitos por columna:')
display(df.isna().sum().to_frame('nulos'))

print('\nCategorías crudas:')
for col in ['idioma', 'recomienda', 'perfil']:
    display(df[col].value_counts(dropna=False).to_frame('n'))

pct_crudo = pd.to_numeric(df['porcentaje_reciente'], errors='coerce')
print('Valores inf en porcentaje_reciente:', int(np.isinf(pct_crudo).sum()))
print('Valores negativos en porcentaje_reciente:', int((pct_crudo < 0).sum()))
print('Valores > 100 en porcentaje_reciente:', int(((pct_crudo > 100) & np.isfinite(pct_crudo)).sum()))

Lo sospechoso, en una lista corta:

1. **Duplicados**: 60 filas idénticas y 20 filas adicionales con un `id_resena` ya presente (80 filas extra en total).
2. **Categorías inconsistentes**: mayúsculas, tildes, espacios y variantes como `"español"`, `"ESPANOL "`, `"si"`, `"NO "`, `"Veterano"`.
3. **Nulos disfrazados**: `"sin dato"` aparece en `perfil` y `votos_utiles`; no es una categoría válida ni un cero.
4. **Tipos**: `votos_utiles` es texto por el `"sin dato"`; `resenas_autor` es decimal solo por los `NaN`; los conteos deberían ser enteros.
5. **Valores imposibles**: horas totales negativas o mayores que el tiempo transcurrido desde el lanzamiento; horas recientes mayores que 14 días; votos negativos; cero reseñas del autor (la reseña actual ya cuenta).
6. **`inf` y porcentajes fuera de rango**: `porcentaje_reciente` se calculó con datos sucios y produce `inf` cuando `horas_totales = 0`.
7. **Cola pesada**: horas y votos tienen distribuciones muy asimétricas; el criterio IQR marcará casos que pueden ser récords reales, no errores.

## 2. Funciones de apoyo

Se reutilizan y adaptan las funciones de la Clase 4 (`resumen_nulos`, `limites_iqr`, `contar_outliers`, `imputar`, `graficar_correlacion`). Además se agregan dos funciones propias: `normalizar_categoria`, que se usa tres veces (idioma, recomendación y perfil), y `resumen_calidad`, que se usa dos veces (original y limpio). Todas las funciones se usan en el notebook.

In [ ]:
def resumen_nulos(datos):
    """Resumen de nulos, porcentaje y tipo por columna (adaptada de Clase 4)."""
    salida = pd.DataFrame({
        'no_nulos': datos.notna().sum(),
        'nulos': datos.isna().sum(),
        'pct_nulos': 100 * datos.isna().mean(),
        'tipo': datos.dtypes.astype(str),
    })
    return salida.sort_values('nulos', ascending=False)


def limites_iqr(serie):
    """Devuelve Q1, Q3, IQR y los límites inferior/superior de Tukey."""
    s = pd.Series(serie).dropna().astype(float)
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return {
        'q1': q1,
        'q3': q3,
        'iqr': iqr,
        'limite_inferior': q1 - 1.5 * iqr,
        'limite_superior': q3 + 1.5 * iqr,
    }


def contar_outliers(serie):
    """Cuenta observaciones fuera de los límites IQR (adaptada de Clase 4)."""
    s = pd.Series(serie).dropna().astype(float)
    lim = limites_iqr(s)
    return int(((s < lim['limite_inferior']) | (s > lim['limite_superior'])).sum())


def imputar(serie, estrategia='mediana'):
    """Imputa con la media o la mediana y devuelve la serie y el valor usado."""
    s = pd.Series(serie)
    if estrategia == 'media':
        valor = s.mean()
    elif estrategia == 'mediana':
        valor = s.median()
    else:
        raise ValueError("estrategia debe ser 'media' o 'mediana'")
    return s.fillna(valor), valor


def graficar_correlacion(matriz, titulo='Matriz de correlación'):
    """Heatmap de una matriz de correlación (adaptada de Clase 4)."""
    fig, ax = plt.subplots(figsize=(11, 8))
    sns.heatmap(
        matriz, annot=True, fmt='.2f', cmap='coolwarm', center=0,
        vmin=-1, vmax=1, square=True, linewidths=0.4, ax=ax,
        cbar_kws={'shrink': 0.8},
    )
    ax.set_title(titulo, fontsize=14, pad=12)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()


def normalizar_categoria(serie, mapa, nombre):
    """Limpia espacios/mayúsculas/tildes y aplica un diccionario explícito."""
    clave = (
        serie.astype('string')
        .str.strip()
        .str.lower()
        .str.normalize('NFKD')
        .str.encode('ascii', 'ignore')
        .str.decode('utf-8')
    )
    resultado = clave.map(mapa)
    no_reconocidos = clave[clave.notna() & ~clave.isin(mapa.keys())].unique()
    if len(no_reconocidos):
        print(f'  Advertencia en {nombre}: valores no reconocidos -> {list(no_reconocidos)}')
    print(f'{nombre}: {serie.notna().sum()} no nulos -> {resultado.notna().sum()} reconocidos; '
          f'{resultado.isna().sum()} quedan nulos')
    return resultado


def resumen_calidad(datos):
    """Métricas comparables entre el archivo original y el trabajado."""
    h = pd.to_numeric(datos['horas_totales'], errors='coerce')
    h2 = pd.to_numeric(datos['horas_2_semanas'], errors='coerce')
    pct = pd.to_numeric(datos['porcentaje_reciente'], errors='coerce')
    votos = pd.to_numeric(datos['votos_utiles'].replace({'sin dato': np.nan}), errors='coerce')
    resenas = pd.to_numeric(datos['resenas_autor'], errors='coerce')
    rec = (
        datos['recomienda'].astype('string').str.strip().str.lower()
        .str.normalize('NFKD').str.encode('ascii', 'ignore').str.decode('utf-8')
    )
    return pd.Series({
        'filas': len(datos),
        'columnas': datos.shape[1],
        'nulos_totales': int(datos.isna().sum().sum()),
        'inf_porcentaje': int(np.isinf(pct).sum()),
        'horas_no_positivas': int((h <= 0).sum()),
        'horas_sobre_maximo': int((h > MAX_HORAS).sum()),
        'horas_2_semanas_sobre_336': int((h2 > MAX_HORAS_2_SEMANAS).sum()),
        'votos_negativos': int((votos < 0).sum()),
        'resenas_autor_cero': int((resenas < 1).sum()),
        'horas_media': h.mean(),
        'horas_mediana': h.median(),
        'pct_recomienda_validos': 100 * (rec == 'si').sum() / rec.notna().sum(),
        'pct_recomienda_ingenuo': 100 * (datos['recomienda'] == 'Sí').sum() / len(datos),
    })

## 3. EDA: diagnóstico y tratamiento

### 3.1 Filas repetidas

Se distinguen dos problemas: filas **idénticas** y filas que repiten la misma `id_resena` pero no son idénticas. La clave `id_resena` debería ser única, así que primero se eliminan los duplicados exactos y luego se conserva una fila por identificador.

In [ ]:
n_exactos = int(df.duplicated().sum())
df_sin_exactos = df.drop_duplicates().copy()

duplicados_id = df_sin_exactos[df_sin_exactos.duplicated(subset=['id_resena'], keep=False)].sort_values('id_resena')
print(f'Duplicados exactos eliminados: {n_exactos}')
print(f'Filas que aún repiten id_resena: {len(duplicados_id)} en {duplicados_id.id_resena.nunique()} grupos')
display(duplicados_id[['id_resena', 'idioma', 'recomienda', 'perfil', 'horas_totales', 'votos_utiles']])

conflictos = duplicados_id.groupby('id_resena').nunique(dropna=False).max(axis=1)
print('Grupos con algún campo distinto:', int((conflictos > 1).sum()))
print('Campos que difieren en esos grupos:')
for c in conflictos.index[conflictos > 1]:
    grupo = duplicados_id[duplicados_id.id_resena == c]
    distintos = [col for col in grupo.columns if grupo[col].nunique(dropna=False) > 1]
    print(f'  id {c}: {distintos}')

df = df_sin_exactos.drop_duplicates(subset=['id_resena'], keep='first').reset_index(drop=True)
print(f'\nFilas después de deduplicar: {len(df)}')
print(f'id_resena únicos: {df.id_resena.nunique()} | rango: {df.id_resena.min()} a {df.id_resena.max()}')

**Decisión.** Se eliminan los 60 duplicados exactos y las 20 filas extra con `id_resena` repetido. En los 20 grupos que no eran idénticos, el único campo que difiere es `votos_utiles`; se conserva la primera aparición porque la clave es la reseña, no el conteo de votos. Esta decisión afecta solo a esos 20 valores de votos; conviene tenerlo como limitación si se analizara la popularidad de las reseñas. El resultado es **3.858 filas**, exactamente una por `id_resena` (1 a 3.858).

### 3.2 Categorías escritas de distintas formas

La limpieza es explícita: se eliminan espacios, se pasa a minúsculas, se quitan tildes y se aplica un diccionario. Así no se inventan categorías nuevas: solo se unifican variantes de la misma categoría.

In [ ]:
MAP_IDIOMA = {'espanol': 'espanol', 'latam': 'latam', 'ingles': 'ingles'}
MAP_RECOMIENDA = {'si': 'Sí', 'no': 'No'}
MAP_PERFIL = {
    'novato': 'novato',
    'intermedio': 'intermedio',
    'veterano': 'veterano',
    'leyenda': 'leyenda',
    'sin dato': np.nan,
}

print('ANTES')
for col in ['idioma', 'recomienda', 'perfil']:
    print(f'\n{col}:')
    display(df[col].value_counts(dropna=False).to_frame('n'))

df['idioma'] = normalizar_categoria(df['idioma'], MAP_IDIOMA, 'idioma')
df['recomienda'] = normalizar_categoria(df['recomienda'], MAP_RECOMIENDA, 'recomienda')
df['perfil'] = normalizar_categoria(df['perfil'], MAP_PERFIL, 'perfil')

print('\nDESPUÉS')
for col in ['idioma', 'recomienda', 'perfil']:
    print(f'\n{col}:')
    display(df[col].value_counts(dropna=False).to_frame('n'))

`idioma` queda en 3 categorías (`espanol`, `latam`, `ingles`); `recomienda` queda en `Sí`/`No` con 20 nulos; `perfil` queda en 4 categorías ordenadas y `"sin dato"` pasa a `NaN` (40 casos), porque **no es una quinta categoría**, es ausencia de información. La normalización es la que permite que el 90,0 % ingenuo de Genovencio se convierta en una tasa calculada sobre respuestas válidas.

### 3.3 Nulos (`NaN`) y nulos disfrazados

Se revisan primero los nulos explícitos y luego los tokens que parecen datos pero significan "sin información". `"sin dato"` no debe convertirse en 0: en `votos_utiles` un 0 significaría "nadie votó útil", que es distinto de "no sé cuántos votos tuvo".

In [ ]:
print('Resumen de nulos después de deduplicar y antes de convertir tipos:')
display(resumen_nulos(df))

print('\nNulos disfrazados en el archivo original:')
for col in ['perfil', 'votos_utiles']:
    n = (df_original[col].astype('string').str.strip().str.lower() == 'sin dato').sum()
    print(f'  {col}: {int(n)} casos con "sin dato"')

df['votos_utiles'] = df['votos_utiles'].replace({'sin dato': np.nan})
for col in ['horas_totales', 'horas_2_semanas', 'porcentaje_reciente', 'resenas_autor', 'votos_utiles']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print('\nResumen de nulos después de convertir a numérico:')
display(resumen_nulos(df))

Se usa `pd.to_numeric(..., errors="coerce")`: convierte lo que se puede y transforma lo que no en `NaN` en vez de detener el notebook. Eso deja `votos_utiles` con 95 nulos (60 `NaN` + 35 `"sin dato"`) y convierte la columna en numérica. `resenas_autor` sigue siendo decimal por los 40 `NaN`, pero sus valores son enteros.

**Decisión:** los nulos de `recomienda` no se imputan porque es la variable de interés; inventar una opinión sería fabricar el resultado. Los de `horas_totales`, `votos_utiles` y `resenas_autor` se dejan como `NaN` y se usan por pares en los análisis. Más adelante sí se imputa `horas_2_semanas` porque es una variable de comportamiento con una distribución muy concentrada en cero y porque se necesita para recalcular `porcentaje_reciente`.

### 3.4 Valores imposibles y columnas con tipo incorrecto

Se usan dos cotas físicas:

* El juego salió el **14-11-2019** y las reseñas se descargaron el **11-10-2026**. Han pasado **2.523 días = 60.552 horas**. Nadie pudo jugar más que eso, ni siquiera 24/7 desde el lanzamiento.
* Dos semanas tienen **14 × 24 = 336 horas**. `horas_2_semanas` no puede superar ese valor.

Además, las horas totales negativas o iguales a cero son incompatibles con el perfil declarado (24 de los 25 ceros tienen perfil distinto de `novato` y 12 tienen horas recientes positivas), los votos no pueden ser negativos y `resenas_autor` no puede ser 0 porque la reseña actual ya cuenta.

In [ ]:
print(f'Máximo físico de horas totales: {MAX_HORAS} h (desde el lanzamiento hasta la descarga)')

diagnostico_imposibles = pd.Series({
    'horas_totales <= 0': int((df.horas_totales <= 0).sum()),
    'horas_totales > 60.552': int((df.horas_totales > MAX_HORAS).sum()),
    'horas_2_semanas > 336': int((df.horas_2_semanas > MAX_HORAS_2_SEMANAS).sum()),
    'votos_utiles < 0': int((df.votos_utiles < 0).sum()),
    'resenas_autor < 1': int((df.resenas_autor < 1).sum()),
    'porcentaje_reciente inf': int(np.isinf(df.porcentaje_reciente).sum()),
    'porcentaje_reciente < 0': int((df.porcentaje_reciente < 0).sum()),
    'porcentaje_reciente > 100': int(((df.porcentaje_reciente > 100) & np.isfinite(df.porcentaje_reciente)).sum()),
})
display(diagnostico_imposibles.to_frame('filas afectadas'))

df.loc[df.horas_totales <= 0, 'horas_totales'] = np.nan
df.loc[df.horas_totales > MAX_HORAS, 'horas_totales'] = np.nan
df.loc[df.horas_2_semanas > MAX_HORAS_2_SEMANAS, 'horas_2_semanas'] = np.nan
df.loc[df.votos_utiles < 0, 'votos_utiles'] = np.nan
df.loc[df.resenas_autor < 1, 'resenas_autor'] = np.nan

inconsistencia_h2 = (
    df.horas_2_semanas.notna() & df.horas_totales.notna() &
    (df.horas_2_semanas > df.horas_totales)
)
print('Filas con horas recientes > horas totales (después de anular imposibles):', int(inconsistencia_h2.sum()))
display(df.loc[inconsistencia_h2, ['id_resena', 'horas_totales', 'horas_2_semanas']])
df.loc[inconsistencia_h2, 'horas_2_semanas'] = df.loc[inconsistencia_h2, 'horas_totales']

df['resenas_autor'] = df['resenas_autor'].round().astype('Int64')
df['votos_utiles'] = df['votos_utiles'].round().astype('Int64')

print('\nDespués de anular imposibles:')
print(f'  horas_totales: min={df.horas_totales.min():.2f}, max={df.horas_totales.max():.2f}, nulos={df.horas_totales.isna().sum()}')
print(f'  horas_2_semanas: min={df.horas_2_semanas.min():.2f}, max={df.horas_2_semanas.max():.2f}, nulos={df.horas_2_semanas.isna().sum()}')
print('  Tipos:')
display(df[['horas_totales', 'horas_2_semanas', 'resenas_autor', 'votos_utiles']].dtypes.to_frame('tipo'))

**Decisión.** Los valores imposibles se reemplazan por `NaN`, no por la media ni por cero, y **no se elimina la fila**: la reseña conserva su idioma, recomendación, perfil y demáscolumnas válidas. En total se anulan 35 horas no positivas, 12 horas sobre el máximo físico, 8 horas recientes sobre 336, 5 votos negativos y 10 conteos de reseñas en cero. Los dos casos con horas recientes apenas mayores que las totales (diferencias de 0,1 h) se corrigen al total, porque probablemente son redondeos.

Los conteos se convierten a `Int64` (entero nullable de pandas): sigue siendo un entero, pero admite `NaN` sin volver a convertirlo en decimal.

### 3.5 `porcentaje_reciente`: `inf`, negativos y valores sobre 100

Esta columna no se imputa: se **recalcula**. Antes conviene comprobar que la fórmula original es exactamente `100 × horas_2_semanas / horas_totales`; si es así, el problema no es la fórmula sino los insumos sucios.

In [ ]:
h1_crudo = pd.to_numeric(df_original.horas_totales, errors='coerce')
h2_crudo = pd.to_numeric(df_original.horas_2_semanas, errors='coerce')
pct_crudo = pd.to_numeric(df_original.porcentaje_reciente, errors='coerce')
calculado_crudo = 100 * h2_crudo / h1_crudo
diferencia = (pct_crudo - calculado_crudo).abs()

print(f'Diferencia máxima entre el porcentaje original y 100*h2/h1 (ignorando inf/NaN): {diferencia[~np.isinf(calculado_crudo)].max():.4f}')
print('Filas con inf, porcentaje negativo o sobre 100:')
problemas_pct = df_original[
    np.isinf(pct_crudo) | (pct_crudo < 0) | ((pct_crudo > 100) & np.isfinite(pct_crudo))
]
display(problemas_pct[['id_resena', 'horas_totales', 'horas_2_semanas', 'porcentaje_reciente']])

La fórmula original coincide hasta el redondeo (diferencia máxima de 0,05 puntos), así que el `inf` viene de dividir por `horas_totales = 0`: 12 casos tienen 0 horas totales y horas recientes positivas. Los 3 porcentajes negativos vienen de horas totales negativas. Los 9 porcentajes sobre 100 vienen de horas recientes mayores que las totales (7 son además mayores que 336 h). La solución es recalcular la columna después de limpiar e imputar, exigiendo `horas_totales > 0`, y acotar el resultado a `[0, 100]`.

### 3.6 Outliers con criterio IQR: ¿errores o récords?

El criterio IQR se aplica **después** de anular los imposibles, porque si se aplicara antes, los errores gigantes contaminarían los cuartiles. En distribuciones tan asimétricas, IQR marca a los jugadores más intensos; la pregunta es si esos valores siguen siendo físicamente posibles.

In [ ]:
variables_outliers = ['horas_totales', 'horas_2_semanas', 'resenas_autor', 'votos_utiles']
filas_iqr = []
for col in variables_outliers:
    s = df[col].dropna().astype(float)
    lim = limites_iqr(s)
    n_out = contar_outliers(s)
    filas_iqr.append({
        'variable': col,
        'Q1': lim['q1'],
        'Q3': lim['q3'],
        'IQR': lim['iqr'],
        'limite_inferior': lim['limite_inferior'],
        'limite_superior': lim['limite_superior'],
        'n_outliers': n_out,
        'pct_outliers': 100 * n_out / len(s),
        'maximo': s.max(),
    })
tabla_iqr = pd.DataFrame(filas_iqr).set_index('variable')
display(tabla_iqr.round(2))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['horas_totales', 'horas_2_semanas', 'votos_utiles']):
    sns.boxplot(x=np.log1p(df[col].dropna().astype(float)), ax=ax, color='#4C72B0')
    ax.set_title(f'{col} (escala log1p)')
    ax.set_xlabel(f'log(1 + {col})')
plt.suptitle('Distribuciones después de anular valores imposibles', y=1.03)
plt.tight_layout()
plt.show()

print('Diez mayores horas totales que quedan (posibles récords, no errores):')
display(df.nlargest(10, 'horas_totales')[['id_resena', 'horas_totales', 'horas_2_semanas']])

El IQR marca **375** horas totales, **524** horas recientes, **460** conteos de reseñas y **604** votos. Son muchos, pero después de anular los imposibles los máximos son plausibles: 12.216 h totales equivalen a unas 4,8 h/día desde el lanzamiento; 289 h en dos semanas equivalen a 20,6 h/día, intenso pero posible; 3.100 votos útiles corresponden a una reseña viral. **No se eliminan**: son récords de la cola derecha, y borrarlos sesgaría el análisis justamente contra los jugadores más intensos. El IQR se usa como diagnóstico, no como regla automática de borrado.

## 4. Imputar o no imputar

Se compara la imputación con **media** y con **mediana** en `horas_2_semanas`, la columna numérica que se necesita para recalcular `porcentaje_reciente`. La distribución tiene mediana 0 y una cola larga: la media está muy influida por pocos jugadores intensos.

In [ ]:
s_h2 = df['horas_2_semanas']
comparacion = []
for estrategia in ['media', 'mediana']:
    serie_imp, valor = imputar(s_h2, estrategia)
    comparacion.append({
        'estrategia': estrategia,
        'valor_imputado': valor,
        'n_imputados': int(s_h2.isna().sum()),
        'media_final': serie_imp.mean(),
        'mediana_final': serie_imp.median(),
        'desv_est_final': serie_imp.std(),
        'suma_final': serie_imp.sum(),
    })
tabla_imputacion = pd.DataFrame(comparacion).set_index('estrategia')
display(tabla_imputacion.round(3))

print(f'Media observada: {s_h2.mean():.3f} h | Mediana observada: {s_h2.median():.3f} h')
print(f'Porcentaje de ceros en los datos observados: {100 * (s_h2 == 0).mean():.1f}%')

La media (≈7,37 h) y la mediana (0 h) cuentan historias distintas. Imputar con la media agregaría 7,37 h de actividad reciente a 98 personas que no la reportaron y aumentaría la suma del grupo; imputar con la mediana agrega 0 h, que es el valor más frecuente y coherente con una mayoría que no jugó en las últimas dos semanas. La media final cambia poco (de 7,19 a 7,37), pero la decisión es más defendible con la **mediana** por la asimetría y la masa de ceros. Se elige la mediana.

In [ ]:
df['horas_2_semanas'], valor_h2 = imputar(df['horas_2_semanas'], 'mediana')
print(f'Valor imputado en horas_2_semanas: {valor_h2}')

df['porcentaje_reciente'] = np.where(
    df['horas_totales'].gt(0),
    (100 * df['horas_2_semanas'] / df['horas_totales']).clip(0, 100),
    np.nan,
)
print('porcentaje_reciente recalculado:')
print(f'  min={df.porcentaje_reciente.min():.2f}, max={df.porcentaje_reciente.max():.2f}, '
      f'inf={int(np.isinf(df.porcentaje_reciente).sum())}, nulos={int(df.porcentaje_reciente.isna().sum())}')
display(df['porcentaje_reciente'].describe().to_frame('porcentaje_reciente'))

Con la mediana, los 98 nulos de `horas_2_semanas` quedan en 0 y `porcentaje_reciente` se puede recalcular sin `inf`. Quedan 157 nulos de porcentaje porque son las mismas filas sin horas totales válidas; no tiene sentido inventar un porcentaje cuando falta el denominador. El máximo es 100 por el acotamiento, no porque todos hayan jugado el 100 % de su tiempo en dos semanas.

**Perfil faltante.** El perfil está definido por reglas explícitas sobre las horas totales. Los 40 `"sin dato"` se pueden reconstruir cuando las horas son válidas, sin inventar: se aplica la misma regla del enunciado. Luego se verifica que, en todas las filas con horas válidas, el perfil coincida con la regla.

In [ ]:
regla_perfil = pd.cut(
    df['horas_totales'],
    bins=[-np.inf, 10, 100, 1000, np.inf],
    labels=['novato', 'intermedio', 'veterano', 'leyenda'],
    right=False,
)
faltantes_perfil = df['perfil'].isna() & regla_perfil.notna()
print('Perfiles faltantes:', int(df['perfil'].isna().sum()))
print('Perfiles reconstruibles con horas válidas:', int(faltantes_perfil.sum()))
df.loc[faltantes_perfil, 'perfil'] = regla_perfil[faltantes_perfil].astype(str)

validos_perfil = df['horas_totales'].notna() & df['perfil'].notna()
coincidencia = (df.loc[validos_perfil, 'perfil'] == regla_perfil[validos_perfil]).mean()
print(f'Coincidencia perfil vs regla en filas con horas válidas: {100 * coincidencia:.2f}%')
print('Perfiles nulos finales:', int(df['perfil'].isna().sum()))
display(df['perfil'].value_counts().to_frame('n'))

Los 40 perfiles faltantes se reconstruyeron y la regla coincide en el **100 %** de las filas con horas totales válidas. Esa validación cruzada respalda la decisión anterior de tratar las horas no positivas como inválidas: cuando las horas son válidas, perfil y horas son perfectamente coherentes; las únicas inconsistencias aparecían justamente en las horas corruptas. No se imputan `horas_totales` (variable central y de cola pesada), `votos_utiles` (no se sabe si el ausente significa 0 o desconocido) ni `resenas_autor` (dato auxiliar); se usan por pares.

## 5. Codificación

Se usa un diccionario explícito para el perfil ordinal y otro para la recomendación binaria. `idioma` se codifica con one-hot (`pd.get_dummies`). Se conservan las columnas originales limpias para que el archivo final sea interpretable.

In [ ]:
PERFIL_ORDINAL = {'novato': 1, 'intermedio': 2, 'veterano': 3, 'leyenda': 4}
RECOMIENDA_BIN = {'No': 0, 'Sí': 1}

df['perfil_ordinal'] = df['perfil'].map(PERFIL_ORDINAL).astype('Int64')
df['recomienda_num'] = df['recomienda'].map(RECOMIENDA_BIN).astype('Int64')

dummies_idioma = pd.get_dummies(df['idioma'], prefix='idioma', dtype=int)
df = pd.concat([df, dummies_idioma], axis=1)

print('Diccionario perfil:', PERFIL_ORDINAL)
print('Diccionario recomienda:', RECOMIENDA_BIN)
print('\nOne-hot de idioma:')
display(dummies_idioma.head())
print('Cada fila suma 1 en los dummies:', bool((dummies_idioma.sum(axis=1) == 1).all()))
print('\nCodificación resultante:')
display(df[['perfil', 'perfil_ordinal', 'recomienda', 'recomienda_num'] + list(dummies_idioma.columns)].head(8))

`recomienda` queda como 1/0 y conserva 20 `NaN` (no se imputa la opinión). `perfil` queda como 1 a 4, respetando el orden. El one-hot usa las tres categorías de idioma; para un modelo convendría eliminar una para evitar colinealidad perfecta, pero para una matriz descriptiva de correlaciones se pueden mostrar las tres y interpretarlas con cuidado.

## 6. Comparar el antes y el después (Teseo)

Se comparan las métricas de calidad del archivo original con el dataset trabajado. Para que la comparación sea justa, los nulos e imposibles se cuentan sobre las columnas de datos; la fila limpia ya incluye las columnas codificadas.

In [ ]:
tabla_teseo = pd.DataFrame([
    resumen_calidad(df_original),
    resumen_calidad(df),
], index=['Original', 'Trabajado']).T

tabla_teseo_fmt = tabla_teseo.copy().astype(object)
enteros = ['filas', 'columnas', 'nulos_totales', 'inf_porcentaje', 'horas_no_positivas',
           'horas_sobre_maximo', 'horas_2_semanas_sobre_336', 'votos_negativos', 'resenas_autor_cero']
for fila in enteros:
    tabla_teseo_fmt.loc[fila] = tabla_teseo_fmt.loc[fila].map(lambda x: f'{int(x):,}')
for fila in ['horas_media', 'horas_mediana', 'pct_recomienda_validos', 'pct_recomienda_ingenuo']:
    tabla_teseo_fmt.loc[fila] = tabla_teseo_fmt.loc[fila].map(lambda x: f'{x:,.2f}')
display(tabla_teseo_fmt)
print(f'Dataset final: {df.shape[0]} filas x {df.shape[1]} columnas')
print('Columnas finales:', list(df.columns))

La tabla muestra el cambio: de **3.938 × 9** a **3.858 × 14**; se eliminaron 80 filas duplicadas y se agregaron 5 columnas codificadas. Los `inf` pasaron de 12 a 0. Los valores imposibles detectados (35 horas no positivas + 12 sobre el máximo + 8 horas recientes sobre 336 + 5 votos negativos + 10 reseñas en cero) se anularon y ya no aparecen como datos válidos. La media de horas cae de **782,04** a **493,32** y la mediana pasa de **152,00** a **153,90**: la media original estaba inflada por los valores extremos y por los errores; la mediana es mucho más estable. El porcentaje de recomendación sobre respuestas válidas es **93,36 %** en el dataset limpio, frente al **89,99 %** que obtenía Genovencio (90,0 % redondeado) contando solo el texto exacto `"Sí"` sobre todas las filas. La versión limpia del cálculo ingenuo, contando `"Sí"` exacto sobre las 3.858 filas, sería **92,87 %**; la diferencia entre 92,87 y 93,36 son los 20 casos sin respuesta.

## 7. Matriz de correlación

Se calculan Pearson y Spearman sobre las variables numéricas y las codificadas. Se elige **Spearman** para interpretar porque: (i) `perfil_ordinal` es ordinal y `recomienda_num` es binaria; (ii) horas, votos y reseñas tienen colas muy largas y outliers legítimos; (iii) la relación entre horas y porcentaje reciente no es lineal. Pearson se calcula para comparar y mostrar cuánto cambia la conclusión cuando se asume linealidad.

In [ ]:
columnas_corr = [
    'horas_totales', 'horas_2_semanas', 'porcentaje_reciente',
    'resenas_autor', 'votos_utiles', 'recomienda_num', 'perfil_ordinal',
    'idioma_espanol', 'idioma_latam', 'idioma_ingles',
]
datos_corr = df[columnas_corr].astype(float)
pearson = datos_corr.corr(method='pearson')
spearman = datos_corr.corr(method='spearman')

display(spearman.round(2))
graficar_correlacion(spearman, 'Correlación de Spearman · dataset limpio')

pares = [
    ('horas_totales', 'horas_2_semanas'),
    ('horas_totales', 'porcentaje_reciente'),
    ('horas_totales', 'votos_utiles'),
    ('resenas_autor', 'votos_utiles'),
    ('recomienda_num', 'horas_totales'),
    ('recomienda_num', 'idioma_ingles'),
]
comparacion_metodos = pd.DataFrame({
    'Pearson': [pearson.loc[a, b] for a, b in pares],
    'Spearman': [spearman.loc[a, b] for a, b in pares],
}, index=[f'{a} ~ {b}' for a, b in pares])
display(comparacion_metodos.round(2))

**Relaciones fuertes.** Las dos más altas son **tautológicas** y no deben interpretarse como hallazgos: `perfil_ordinal ~ horas_totales` (Spearman ≈ 0,92) porque el perfil se construye con las horas, y `horas_2_semanas ~ porcentaje_reciente` (≈ 0,96) porque el porcentaje se calcula con las horas recientes. Entre las relaciones no tautológicas, la más fuerte es **horas_totales ~ horas_2_semanas** (≈ 0,37): quienes acumulan más horas también tienden a jugar más en las últimas dos semanas, pero la relación está lejos de ser determinista. Le siguen **resenas_autor ~ votos_utiles** (≈ 0,32): autores con más reseñas tienden a recibir más votos útiles, y **votos_utiles ~ idioma_ingles** (≈ 0,32).

**Relaciones débiles o ausentes.** `recomienda_num` con horas totales (≈ -0,07), con horas recientes (≈ 0,01) y con perfil (≈ -0,05) es prácticamente cero. Es decir, **no hay evidencia de que a los que más juegan les guste más el juego**. También son débiles las correlaciones entre `horas_totales` e idioma (`-0,05` a `0,04`). La correlación negativa entre `recomienda_num` e `idioma_ingles` (≈ -0,19) anticipa la diferencia por idioma que se ve abajo.

**Pearson vs Spearman.** Hay cambios de signo que muestran por qué no conviene usar Pearson a ciegas: `horas_totales ~ porcentaje_reciente` pasa de -0,10 (Pearson) a +0,21 (Spearman), y `horas_totales ~ votos_utiles` pasa de 0,02 a 0,17. Los outliers y la no linealidad distorsionan Pearson. Por eso se interpreta Spearman, que es más adecuado para variables ordinales, binarias y colas pesadas.

## 8. Responderle a Genovencio

Se recalculan sus dos titulares. El primero se compara con su método original (contar solo `"Sí"` exacto sobre las 3.938 filas). El segundo se resume con mediana y cuartiles, porque la media es muy sensible a los récords. Luego se compara la recomendación por perfil y por cuartiles de horas.

In [ ]:
pct_genovencio = 100 * (df_original['recomienda'] == 'Sí').mean()
validos_rec = int(df['recomienda_num'].notna().sum())
pct_limpio = 100 * df['recomienda_num'].mean()

print(f'Titular original de Genovencio (solo "Sí" exacto / 3.938): {pct_genovencio:.1f}%')
print(f'Titular limpio (Sí / respuestas válidas): {pct_limpio:.1f}% '
      f'({int(df.recomienda_num.sum())}/{validos_rec})')
print(f'Cálculo ingenuo sobre el dataset limpio ("Sí" exacto / 3.858): '
      f'{100 * (df.recomienda == "Sí").mean():.1f}%')

print(f'\nHoras típicas: mediana = {df.horas_totales.median():.1f} h; '
      f'media = {df.horas_totales.mean():.1f} h; n = {df.horas_totales.count()}')
print('Cuartiles de horas totales:')
display(df.horas_totales.quantile([0.25, 0.5, 0.75]).to_frame('horas'))

tabla_idioma = (
    df.groupby('idioma', observed=True)
    .agg(
        resenas=('id_resena', 'size'),
        validos=('recomienda_num', 'count'),
        pct_recomienda=('recomienda_num', lambda s: 100 * s.mean()),
        mediana_horas=('horas_totales', 'median'),
    )
    .sort_values('pct_recomienda', ascending=False)
)
print('\nRecomendación por idioma:')
display(tabla_idioma.round(2))

tabla_perfil = (
    df.groupby('perfil', observed=True)
    .agg(
        resenas=('id_resena', 'size'),
        validos=('recomienda_num', 'count'),
        pct_recomienda=('recomienda_num', lambda s: 100 * s.mean()),
        mediana_horas=('horas_totales', 'median'),
    )
    .reindex(['novato', 'intermedio', 'veterano', 'leyenda'])
)
print('\nRecomendación por perfil:')
display(tabla_perfil.round(2))

tmp = df[df.horas_totales.notna()].copy()
tmp['cuartil_horas'] = pd.qcut(tmp.horas_totales, q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
tabla_cuartiles = (
    tmp.groupby('cuartil_horas', observed=True)
    .agg(
        resenas=('id_resena', 'size'),
        pct_recomienda=('recomienda_num', lambda s: 100 * s.mean()),
        mediana_horas=('horas_totales', 'median'),
        min_horas=('horas_totales', 'min'),
        max_horas=('horas_totales', 'max'),
    )
)
print('\nRecomendación por cuartiles de horas totales:')
display(tabla_cuartiles.round(2))

print('\nHoras según recomendación:')
display(df.groupby('recomienda', observed=True)['horas_totales'].agg(['count', 'mean', 'median']).round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
tabla_idioma['pct_recomienda'].plot(kind='bar', ax=axes[0], color='#4C72B0')
axes[0].set_title('Porcentaje que recomienda, por idioma')
axes[0].set_ylabel('%')
axes[0].set_ylim(80, 100)
axes[0].tick_params(axis='x', rotation=0)
tabla_perfil['pct_recomienda'].plot(kind='bar', ax=axes[1], color='#55A868')
axes[1].set_title('Porcentaje que recomienda, por perfil')
axes[1].set_ylabel('%')
axes[1].set_ylim(80, 100)
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

### Respuesta a Genovencio

**Qué cambió.** Su 90,0 % no era una tasa: era el número de filas que decían exactamente `"Sí"` dividido por todas las filas, incluyendo 20 reseñas sin respuesta y decenas de variantes como `"SI"`, `"sí"` o `" Sí"`. Al normalizar y deduplicar, la tasa válida es **93,4 %** (3.583 de 3.838 respuestas; quedan 20 sin dato). Su promedio de 782 horas estaba inflado por 12 valores por encima del máximo físico, 35 horas no positivas y una cola de récords; la mediana limpia es **153,9 h** (IQR 49,1–502,4), mucho más representativa de un jugador típico. La media limpia sigue siendo alta (493,3 h) precisamente porque los récords son reales; por eso conviene publicar la mediana, no el promedio.

**Qué puede publicar.** Puede publicar que la enorme mayoría recomienda el juego, con una diferencia marcada por idioma: **96,8 %** en español, **97,2 %** en latam y **86,9 %** en inglés. También puede publicar que la mediana de horas es de unas **154 h** y que la mitad de los jugadores tiene entre 49 y 502 horas. Lo que **no** puede publicar es que "a los que más juegan les gusta más": la correlación entre horas y recomendación es ≈ **-0,07**, y por perfil las tasas son 90,0 % (novato), 95,1 % (intermedio), 93,8 % (veterano) y 88,7 % (leyenda). En los cuartiles de horas, el cuartil más alto es el que menos recomienda (≈ 89,6 %), y quienes no recomiendan tienen una mediana de horas mayor que quienes sí recomiendan (278,5 h vs 148,8 h). Si hay un patrón, es que los más veteranos son algo más críticos, no más entusiastas.

**Qué implicó limpiar.** Sin limpieza, los duplicados inflaban el conteo, las variantes de texto escondían recomendaciones y los valores imposibles distorsionaban la media y creaban `inf`. La limpieza no cambió la dirección general (el juego se recomienda masivamente), pero sí cambió los números que se pueden defender: 93,4 % en vez de 90,0 %, y 154 h típicas en vez de 782 h promedio. Además, cualquier conclusión sobre horas y gusto es observacional: son reseñas de Steam, no una muestra aleatoria, y la recomendación puede reflejar crítica al estado del juego, no cuánto lo juegan. Es importante decirlo en la columna.

## 9. Guardar el dataset limpio

Se guarda el dataset final en CSV y Excel, sin el índice y con nombres distintos al original. El CSV se escribe en UTF-8. Antes de cerrar se vuelve a leer cada archivo para comprobar dimensiones y que no se haya guardado una columna de índice.

In [ ]:
columnas_finales = [
    'id_resena', 'idioma', 'recomienda', 'perfil',
    'horas_totales', 'horas_2_semanas', 'porcentaje_reciente',
    'resenas_autor', 'votos_utiles',
    'perfil_ordinal', 'recomienda_num',
    'idioma_espanol', 'idioma_latam', 'idioma_ingles',
]
df_final = df[columnas_finales].copy()

df_final.to_csv('aoe2_trabajado.csv', index=False, encoding='utf-8')
df_final.to_excel('aoe2_trabajado.xlsx', index=False, engine='openpyxl')

for col in ['horas_totales', 'horas_2_semanas', 'porcentaje_reciente']:
    assert not np.isinf(df_final[col].dropna()).any(), f'inf en {col}'

csv_leido = pd.read_csv('aoe2_trabajado.csv')
xlsx_leido = pd.read_excel('aoe2_trabajado.xlsx')
print('CSV guardado:', csv_leido.shape, '| columnas:', len(csv_leido.columns))
print('Excel guardado:', xlsx_leido.shape, '| columnas:', len(xlsx_leido.columns))
print('¿Hay columna de índice?', 'Unnamed: 0' in csv_leido.columns or 'Unnamed: 0' in xlsx_leido.columns)
print('Tamaños (KB):', round(os.path.getsize('aoe2_trabajado.csv') / 1024, 1),
      '|', round(os.path.getsize('aoe2_trabajado.xlsx') / 1024, 1))
display(df_final.head(5))

**Resultado final:** 3.858 filas × 14 columnas. Los archivos `aoe2_trabajado.csv` y `aoe2_trabajado.xlsx` se guardaron sin índice en la carpeta del notebook. Las 14 columnas son las 9 originales limpias más `perfil_ordinal`, `recomienda_num` y los tres dummies de idioma.

**Limitaciones que conviene declarar:** 20 reseñas duplicadas diferían solo en `votos_utiles` y se conservó la primera; 157 filas quedaron sin horas totales válidas y, por lo tanto, sin porcentaje reciente; 20 reseñas no tienen recomendación y no se imputaron; `votos_utiles` y `resenas_autor` conservan nulos y se usan por pares. Ninguna de estas decisiones cambia la respuesta principal, pero sí deben quedar explícitas.

## Declaración de uso de inteligencia artificial

Para esta resolución se utilizó **opencode** como asistente de IA para: (i) revisar sistemáticamente el CSV y detectar patrones de suciedad; (ii) proponer y depurar el código de limpieza, imputación, codificación y visualización; y (iii) redactar borradores de interpretación. Todas las decisiones fueron verificadas ejecutando el notebook completo y contrastando los resultados con los datos. El criterio, las justificaciones y la defensa oral son responsabilidad del estudiante.